# Phase 1 Kaggle Full - PDF-Extract-Kit + DeepSeek-OCR + Qwen Correction

Pipeline đầy đủ cho Phase 1: render PDF, detect layout bằng PDF-Extract-Kit/DocLayout-YOLO, crop ảnh/bảng/caption, OCR tiếng Việt bằng DeepSeek-OCR, hậu xử lý lỗi OCR bằng Qwen local, rebuild `book.md`, `book.docx`, `rag_chunks.jsonl`, metadata và quality report.

In [ ]:
from pathlib import Path
import json, math, os, re, subprocess, sys, zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTHONUTF8', '1')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Thay đường dẫn input của file PDF vào đây.
INPUT_PDF = Path('/kaggle/input/vietnam-schoolbooks/SGK Lịch sử và địa lí 6 CD.pdf')
if not INPUT_PDF.exists() and Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob('*.pdf'))
    if candidates:
        INPUT_PDF = candidates[0]
if not INPUT_PDF.exists():
    INPUT_PDF = Path('books/SGK Lịch sử và địa lí 6 CD.pdf')

BASE_OUTPUT_DIR = Path('/kaggle/working/class_6_phase1_full') if Path('/kaggle/working').exists() else Path('extracted/class_6_phase1_full')
CACHE_DIR = Path('/kaggle/working/_phase1_shared_cache') if Path('/kaggle/working').exists() else Path('extracted/_phase1_shared_cache')
OUTPUT_DIR = BASE_OUTPUT_DIR
KIT_DIR = OUTPUT_DIR / '_PDF-Extract-Kit'
MODEL_DIR = OUTPUT_DIR / '_models_pdf_extract_kit'
PAGES_DIR = OUTPUT_DIR / 'pages'
IMAGES_DIR = OUTPUT_DIR / 'images'
CAPTIONS_DIR = OUTPUT_DIR / 'caption_crops'
METADATA_DIR = OUTPUT_DIR / 'metadata'
PREVIEW_DIR = OUTPUT_DIR / '_previews'
LAYOUT_VIS_DIR = PREVIEW_DIR / 'layout_boxes'
TEXT_CROPS_DIR = OUTPUT_DIR / 'text_crops'
DEEPSEEK_OUT_DIR = OUTPUT_DIR / '_deepseek_pages'
BLOCK_OCR_OUT_DIR = OUTPUT_DIR / '_deepseek_blocks'
MANIFEST_DIR = OUTPUT_DIR / '_manifests'

# Test trước pages 6-10. Khi ổn thì đổi START_PAGE=1, END_PAGE=None để chạy toàn bộ sách.
START_PAGE = 6
END_PAGE = 10
RENDER_SCALE = 2.0

END_PAGE_LABEL = 'end' if END_PAGE is None else f'{END_PAGE:03d}'
PART_NAME = f'pages_{START_PAGE:03d}_{END_PAGE_LABEL}'
OUTPUT_DIR = BASE_OUTPUT_DIR / PART_NAME
KIT_DIR = CACHE_DIR / 'PDF-Extract-Kit'
MODEL_DIR = CACHE_DIR / 'models_pdf_extract_kit'
PAGES_DIR = OUTPUT_DIR / 'pages'
IMAGES_DIR = OUTPUT_DIR / 'images'
CAPTIONS_DIR = OUTPUT_DIR / 'caption_crops'
TEXT_CROPS_DIR = OUTPUT_DIR / 'text_crops'
METADATA_DIR = OUTPUT_DIR / 'metadata'
PREVIEW_DIR = OUTPUT_DIR / '_previews'
LAYOUT_VIS_DIR = PREVIEW_DIR / 'layout_boxes'
DEEPSEEK_OUT_DIR = OUTPUT_DIR / '_deepseek_pages'
BLOCK_OCR_OUT_DIR = OUTPUT_DIR / '_deepseek_blocks'
MANIFEST_DIR = OUTPUT_DIR / '_manifests'

LAYOUT_MODEL_REPO_CANDIDATES = ['opendatalab/pdf-extract-kit-1.0', 'opendatalab/PDF-Extract-Kit-1.0']
LAYOUT_WEIGHT_PATTERN = 'models/Layout/YOLO/doclayout_yolo_ft.pt'
IMG_SIZE = 1024
CONF_THRES = 0.25
IOU_THRES = 0.45
DEVICE = 'auto'  # 'auto', '0', 'cpu'

VISUAL_TYPES = {'figure', 'table'}
CAPTION_TYPES = {'figure_caption', 'table_caption'}
SAVE_CAPTION_CROPS = True
INCLUDE_CAPTION_CROPS_IN_MD = False
MIN_CROP_AREA_RATIO = 0.002
PAD_PX = 8
RUN_DEEPSEEK_OCR = True
DEEPSEEK_MODEL = 'deepseek-ai/DeepSeek-OCR'
DEEPSEEK_PROMPT = '<image>\n<|grounding|>Convert the textbook page to clean Vietnamese markdown. Preserve headings, numbered tasks, captions, and table text. Do not invent missing text.'
DEEPSEEK_ATTN_IMPL = 'eager'
DEEPSEEK_BASE_SIZE = 1024
DEEPSEEK_IMAGE_SIZE = 640
DEEPSEEK_CROP_MODE = True
DEEPSEEK_TEST_COMPRESS = True
PAGE_OCR_SAVE_RESULTS = True
DEEPSEEK_CPU_BFLOAT16 = True
GPU_IDS_TO_USE = 'auto'       # 'auto', '0', '0,1'
MAX_PARALLEL_GPUS = 1         # Test 6-10 nên để 1. Khi ổn có thể đổi 2 trên T4 x2.
ALLOW_CPU_DEEPSEEK = False
FORCE_DEEPSEEK = False

FORCE_BLOCK_OCR = False       # Bật True khi cần ghi đè _deepseek_blocks lỗi trong cùng session.
FORCE_PAGE_OCR = False        # Bật True khi cần ghi đè _deepseek_pages.

RUN_BLOCK_OCR = False         # DeepSeek-OCR không ổn định trên crop nhỏ: dễ division by zero. Chỉ bật để thử nghiệm.
RUN_PAGE_OCR = True           # Nguồn OCR chính/fallback, sau đó phân bổ text vào layout blocks.
USE_PDF_TEXT_LAYER_FOR_BLOCKS = True
USE_PAGE_OCR_TO_LAYOUT_BLOCKS = True
TEXT_OCR_TYPES = {'title', 'plain text', 'figure_caption', 'table_caption', 'table_footnote', 'formula_caption', 'table'}
BLOCK_OCR_PROMPT = '<image>\nExtract only the visible Vietnamese text from this layout block. Preserve line breaks, headings, bullets, table rows, numbers, and captions. Do not describe images and do not invent missing text.'
BLOCK_OCR_BASE_SIZE = 768
BLOCK_OCR_IMAGE_SIZE = 640
BLOCK_OCR_CROP_MODE = False
BLOCK_OCR_TEST_COMPRESS = True
BLOCK_OCR_SAVE_RESULTS = False  # Tránh lỗi division by zero ở postprocess grounding của crop nhỏ.
TEXT_CROP_PAD_PX = 10
TEXT_CROP_MIN_WIDTH = 512
TEXT_CROP_MIN_HEIGHT = 192
TEXT_CROP_MAX_UPSCALE = 3.0
MIN_TEXT_CROP_AREA_RATIO = 0.00003
MIN_BLOCK_TEXT_CHARS = 2
MIN_BLOCK_OCR_TEXT_COVERAGE = 0.55

RUN_LLM_SPELLCHECK = False  # Layout-first mặc định tắt để không làm xáo trộn thứ tự bbox.
SPELLCHECK_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
SPELLCHECK_MAX_SECTION_CHARS = 4200
SPELLCHECK_MAX_NEW_TOKENS = 4096
SPELLCHECK_GPU_IDS_TO_USE = 'auto'
SPELLCHECK_MAX_PARALLEL_GPUS = 1  # Khi ổn có thể đổi 2 trên T4 x2.
SPELLCHECK_OVERWRITE_BOOK_MD = True
SPELLCHECK_STRICT = False
FORCE_SPELLCHECK = False

MIN_PAGE_TEXT_CHARS = 80
MAX_SHORT_PAGE_RATIO = 0.35
MAX_EMPTY_PAGE_RATIO = 0.15
QUALITY_FAIL_ON_EMPTY_TEXT = True
QUALITY_STRICT = True

INCLUDE_RENDERED_PAGES_IN_ZIP = False
INCLUDE_LAYOUT_BOXES_IN_ZIP = False
INCLUDE_RAW_WORKER_OUTPUTS_IN_ZIP = False
DELETE_INTERMEDIATE_DIRS_AFTER_ZIP = True
DELETE_SHARED_CACHE_AFTER_RUN = False

ZIP_OUTPUT = True
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200

for directory in [BASE_OUTPUT_DIR, CACHE_DIR, OUTPUT_DIR, MODEL_DIR, PAGES_DIR, IMAGES_DIR, CAPTIONS_DIR, TEXT_CROPS_DIR, METADATA_DIR, PREVIEW_DIR, LAYOUT_VIS_DIR, DEEPSEEK_OUT_DIR, BLOCK_OCR_OUT_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('INPUT_PDF =', INPUT_PDF)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Page range =', START_PAGE, END_PAGE)


In [ ]:
def run_cmd(cmd, *, env=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    sys.stdout.flush()
    return subprocess.run(list(map(str, cmd)), env=env, check=check)

# Cài layout-only, tránh requirements.txt đầy đủ vì nó kéo PaddleOCR/lmdeploy và CUDA stack nặng.
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'PyMuPDF', 'Pillow', 'opencv-python-headless', 'numpy<2.5',
    'huggingface_hub', 'ultralytics>=8.2.85',
    'omegaconf', 'pyyaml', 'python-docx', 'tqdm'
])

if RUN_DEEPSEEK_OCR or RUN_LLM_SPELLCHECK:
    # DeepSeek-OCR/Qwen deps. Không cài torch/torchvision để tránh kéo lệch CUDA stack của Kaggle.
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        'transformers==4.46.3', 'tokenizers==0.20.3', 'img2pdf',
        'einops', 'easydict', 'addict', 'accelerate', 'safetensors'
    ])
doclayout_install = None
for package in ['doclayout-yolo==0.0.4', 'doclayout-yolo==0.0.3', 'doclayout-yolo==0.0.2b1']:
    doclayout_install = run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', package], check=False)
    if doclayout_install.returncode == 0:
        print('doclayout-yolo installed:', package)
        break
if doclayout_install is None or doclayout_install.returncode != 0:
    print('doclayout-yolo install failed; notebook will fallback to ultralytics.YOLO.')

try:
    import torch
    GPU_COUNT = torch.cuda.device_count()
    GPU_IDS = list(range(GPU_COUNT))
except Exception as exc:
    print('torch import failed:', repr(exc))
    GPU_COUNT = 0
    GPU_IDS = []

CPU_COUNT = os.cpu_count() or 2
os.environ['OMP_NUM_THREADS'] = str(CPU_COUNT)
os.environ['MKL_NUM_THREADS'] = str(CPU_COUNT)
print('CPU_COUNT =', CPU_COUNT)
print('GPU_IDS =', GPU_IDS)
run_cmd(['nvidia-smi'], check=False)
if RUN_DEEPSEEK_OCR and not GPU_IDS and not ALLOW_CPU_DEEPSEEK:
    raise RuntimeError('No CUDA GPU found. Bật GPU trong Kaggle Notebook Settings trước khi chạy DeepSeek-OCR.')


In [ ]:
# Clone PDF-Extract-Kit để dùng trực tiếp model/code layout, không dùng MinerU wrapper.
if not KIT_DIR.exists():
    run_cmd(['git', 'clone', '--depth', '1', 'https://github.com/opendatalab/PDF-Extract-Kit.git', str(KIT_DIR)], check=True)
else:
    print('PDF-Extract-Kit repo exists:', KIT_DIR)

if str(KIT_DIR) not in sys.path:
    sys.path.insert(0, str(KIT_DIR))

from huggingface_hub import snapshot_download

weight_path = MODEL_DIR / LAYOUT_WEIGHT_PATTERN
if not weight_path.exists():
    last_error = None
    for repo_id in LAYOUT_MODEL_REPO_CANDIDATES:
        try:
            print('download layout weights from', repo_id)
            snapshot_download(repo_id=repo_id, local_dir=str(MODEL_DIR), allow_patterns=[LAYOUT_WEIGHT_PATTERN], max_workers=8)
            if weight_path.exists():
                break
        except Exception as exc:
            last_error = exc
            print('download failed:', repr(exc))
    if not weight_path.exists():
        raise RuntimeError(f'Cannot download layout weight: {last_error}')
print('weight_path =', weight_path, weight_path.stat().st_size)


In [ ]:
import fitz
from tqdm.auto import tqdm


def render_one_page(args):
    pdf_path, page_index, scale, out_dir = args
    doc = fitz.open(str(pdf_path))
    page_no = page_index + 1
    out_path = Path(out_dir) / f'page_{page_no:03d}.jpg'
    if not out_path.exists():
        pix = doc[page_index].get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        pix.save(str(out_path))
    return {'page_number': page_no, 'path': str(out_path)}


pdf_doc = fitz.open(str(INPUT_PDF))
total_pages = len(pdf_doc)
start_idx = max(0, START_PAGE - 1)
end_idx = total_pages if END_PAGE is None else min(total_pages, END_PAGE)
page_indexes = list(range(start_idx, end_idx))
print('total_pages =', total_pages, 'selected =', len(page_indexes))

render_jobs = [(INPUT_PDF, i, RENDER_SCALE, PAGES_DIR) for i in page_indexes]
page_images = []
workers = min(CPU_COUNT, max(1, len(render_jobs)))
with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(render_one_page, job) for job in render_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc='Render PDF pages'):
        page_images.append(future.result())
page_images = sorted(page_images, key=lambda item: item['page_number'])
print('rendered pages:', len(page_images))


In [ ]:
from PIL import Image, ImageDraw

ID_TO_NAME = {0: 'title', 1: 'plain text', 2: 'abandon', 3: 'figure', 4: 'figure_caption', 5: 'table', 6: 'table_caption', 7: 'table_footnote', 8: 'isolate_formula', 9: 'formula_caption'}
COLORS = {'title': '#2F80ED', 'plain text': '#27AE60', 'abandon': '#828282', 'figure': '#EB5757', 'figure_caption': '#F2994A', 'table': '#9B51E0', 'table_caption': '#BB6BD9', 'table_footnote': '#56CCF2', 'isolate_formula': '#219653', 'formula_caption': '#6FCF97'}


def resolve_device():
    if DEVICE != 'auto':
        return DEVICE
    try:
        import torch
        return '0' if torch.cuda.is_available() else 'cpu'
    except Exception:
        return 'cpu'


def load_layout_model():
    config = {
        'model_path': str(weight_path),
        'img_size': IMG_SIZE,
        'conf_thres': CONF_THRES,
        'iou_thres': IOU_THRES,
        'visualize': False,
        'device': LAYOUT_DEVICE,
    }
    try:
        from pdf_extract_kit.tasks.layout_detection.models.yolo import LayoutDetectionYOLO
        print('Using PDF-Extract-Kit LayoutDetectionYOLO wrapper')
        return 'pdf_extract_kit.LayoutDetectionYOLO', LayoutDetectionYOLO(config)
    except Exception as exc:
        print('PDF-Extract-Kit wrapper failed, fallback to direct YOLO:', repr(exc))

    try:
        from doclayout_yolo import YOLOv10
        print('Using doclayout_yolo.YOLOv10 fallback')
        return 'doclayout_yolo.YOLOv10', YOLOv10(str(weight_path))
    except Exception as exc:
        print('YOLOv10 load failed, fallback to ultralytics.YOLO:', repr(exc))
        from ultralytics import YOLO
        return 'ultralytics.YOLO', YOLO(str(weight_path))


LAYOUT_DEVICE = resolve_device()
layout_engine, layout_model = load_layout_model()
print('LAYOUT_DEVICE =', LAYOUT_DEVICE)
print('layout_engine =', layout_engine)


def predict_one_layout(page_image_path):
    page_image_path = str(page_image_path)
    if layout_engine == 'pdf_extract_kit.LayoutDetectionYOLO':
        return layout_model.predict([page_image_path], str(LAYOUT_VIS_DIR), image_ids=[Path(page_image_path).stem])[0]
    return layout_model.predict(page_image_path, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False, device=LAYOUT_DEVICE)[0]


def result_to_detections(result):
    boxes_obj = result.boxes
    if boxes_obj is None or len(boxes_obj) == 0:
        return []
    xyxy = boxes_obj.xyxy.detach().cpu().numpy()
    cls = boxes_obj.cls.detach().cpu().numpy().astype(int)
    conf = boxes_obj.conf.detach().cpu().numpy()
    dets = []
    for idx, (box, klass, score) in enumerate(zip(xyxy, cls, conf)):
        x0, y0, x1, y1 = [float(v) for v in box]
        label = ID_TO_NAME.get(int(klass), str(int(klass)))
        dets.append({'det_index': idx, 'category_id': int(klass), 'category_type': label, 'score': float(score), 'bbox': [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)], 'poly': [round(x0, 2), round(y0, 2), round(x1, 2), round(y0, 2), round(x1, 2), round(y1, 2), round(x0, 2), round(y1, 2)], 'source': 'pdf_extract_kit_doclayout_yolo'})
    return sorted(dets, key=lambda d: (d['bbox'][1], d['bbox'][0]))


def draw_layout(page_image_path, detections, out_path):
    image = Image.open(page_image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    for det in detections:
        x0, y0, x1, y1 = det['bbox']
        label = det['category_type']
        color = COLORS.get(label, '#000000')
        draw.rectangle((x0, y0, x1, y1), outline=color, width=3)
        draw.text((x0 + 3, max(0, y0 - 14)), f'{label} {det["score"]:.2f}', fill=color)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(out_path, quality=90)


layout_by_page = {}
for page in tqdm(page_images, desc='PDF-Extract-Kit layout detection'):
    page_no = int(page['page_number'])
    result = predict_one_layout(page['path'])
    detections = result_to_detections(result)
    layout_by_page[page_no] = detections
    draw_layout(page['path'], detections, LAYOUT_VIS_DIR / f'page_{page_no:03d}_layout.jpg')
    counts = {k: sum(1 for d in detections if d['category_type'] == k) for k in sorted(VISUAL_TYPES | CAPTION_TYPES)}
    print('page', page_no, 'detections', len(detections), counts)


In [ ]:
def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '').replace('\u00a0', ' ')).strip()


def bbox_area(bbox):
    x0, y0, x1, y1 = bbox
    return max(0.0, x1 - x0) * max(0.0, y1 - y0)


def bbox_center(bbox):
    x0, y0, x1, y1 = bbox
    return ((x0 + x1) / 2, (y0 + y1) / 2)


def clamp_bbox(bbox, width, height, pad=0):
    x0, y0, x1, y1 = [int(round(v)) for v in bbox]
    return (max(0, x0 - pad), max(0, y0 - pad), min(width, x1 + pad), min(height, y1 + pad))


def nearest_caption(visual, captions, used_caption_indexes, page_width, page_height):
    vx, _ = bbox_center(visual['bbox'])
    vbox = visual['bbox']
    target_type = 'table_caption' if visual['category_type'] == 'table' else 'figure_caption'
    candidates = []
    for idx, cap in enumerate(captions):
        if idx in used_caption_indexes or cap['category_type'] != target_type:
            continue
        cx, _ = bbox_center(cap['bbox'])
        horizontal = abs(cx - vx) / max(1, page_width)
        if cap['bbox'][1] >= vbox[3]:
            vertical_gap = cap['bbox'][1] - vbox[3]
            direction_penalty = 0.0
        else:
            vertical_gap = vbox[1] - cap['bbox'][3]
            direction_penalty = 0.12
        if horizontal > 0.38 or vertical_gap > page_height * 0.22:
            continue
        score = horizontal * 1.6 + vertical_gap / max(1, page_height) + direction_penalty
        candidates.append((score, idx, cap))
    if not candidates:
        return None, None
    _, idx, cap = min(candidates, key=lambda item: item[0])
    used_caption_indexes.add(idx)
    return idx, cap


image_records = []
block_records = []
page_records = []
for page in page_images:
    page_no = int(page['page_number'])
    page_image_path = Path(page['path'])
    image = Image.open(page_image_path).convert('RGB')
    width, height = image.size
    detections = layout_by_page.get(page_no, [])
    min_area = width * height * MIN_CROP_AREA_RATIO
    captions = [d for d in detections if d['category_type'] in CAPTION_TYPES]
    used_caption_indexes = set()
    page_blocks = []
    visual_index = 1

    for order, det in enumerate(detections):
        block = dict(det)
        block.update({'page': page_no, 'page_number': page_no, 'order': order})
        if det['category_type'] in VISUAL_TYPES and bbox_area(det['bbox']) >= min_area:
            _, caption = nearest_caption(det, captions, used_caption_indexes, width, height)
            visual_type = det['category_type']
            stem = f'page_{page_no:03d}_{visual_type}_{visual_index:02d}'
            x0, y0, x1, y1 = clamp_bbox(det['bbox'], width, height, pad=PAD_PX)
            crop_path = IMAGES_DIR / f'{stem}.jpg'
            image.crop((x0, y0, x1, y1)).save(crop_path, quality=92)
            caption_crop_rel = ''
            caption_bbox = None
            if caption and SAVE_CAPTION_CROPS:
                cx0, cy0, cx1, cy1 = clamp_bbox(caption['bbox'], width, height, pad=PAD_PX)
                caption_crop = CAPTIONS_DIR / f'{stem}_caption.jpg'
                image.crop((cx0, cy0, cx1, cy1)).save(caption_crop, quality=92)
                caption_crop_rel = caption_crop.relative_to(OUTPUT_DIR).as_posix()
                caption_bbox = caption['bbox']
            block.update({'type': 'table' if visual_type == 'table' else 'image', 'id': stem, 'label': stem, 'path': crop_path.relative_to(OUTPUT_DIR).as_posix(), 'caption': '', 'caption_bbox': caption_bbox, 'caption_crop_path': caption_crop_rel})
            image_records.append(block)
            visual_index += 1
        else:
            block.update({'type': det['category_type']})
        page_blocks.append(block)
        block_records.append(block)

    try:
        page_text = clean_text(pdf_doc[page_no - 1].get_text('text'))
    except Exception:
        page_text = ''
    page_record = {'source_pdf': str(INPUT_PDF), 'page_number': page_no, 'page_image_path': page_image_path.relative_to(OUTPUT_DIR).as_posix(), 'width': width, 'height': height, 'text': page_text, 'layout_dets': detections, 'text_blocks': page_blocks, 'engine': 'pdf_extract_kit_doclayout_yolo'}
    page_records.append(page_record)
    (METADATA_DIR / 'pages').mkdir(parents=True, exist_ok=True)
    (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')

(METADATA_DIR / 'layout_detections.json').write_text(json.dumps(page_records, ensure_ascii=False, indent=2), encoding='utf-8')
(METADATA_DIR / 'images.json').write_text(json.dumps(image_records, ensure_ascii=False, indent=2), encoding='utf-8')
(METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + '\n', encoding='utf-8')
print('pages:', len(page_records), 'images/tables:', len(image_records), 'blocks:', len(block_records))


In [ ]:
# Chuẩn layout-first: crop từng text/caption/table block từ bbox của DocLayout-YOLO.
TEXT_BLOCK_TYPES = set(TEXT_OCR_TYPES)
TEXT_BLOCK_MANIFEST = MANIFEST_DIR / 'deepseek_text_blocks.jsonl'


def bbox_iou(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    if inter <= 0:
        return 0.0
    return inter / max(1.0, bbox_area(a) + bbox_area(b) - inter)


def bbox_containment(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    return inter / max(1.0, min(bbox_area(a), bbox_area(b)))


def block_priority(block):
    priority = {
        'title': 6,
        'table_caption': 5,
        'figure_caption': 5,
        'plain text': 4,
        'table': 3,
        'table_footnote': 2,
        'formula_caption': 1,
    }
    return priority.get(block.get('category_type') or block.get('type'), 0)


def dedupe_layout_text_candidates(blocks):
    candidates = []
    for block in blocks:
        layout_type = block.get('category_type') or block.get('type')
        if layout_type not in TEXT_BLOCK_TYPES:
            continue
        bbox = block.get('bbox')
        if not bbox or bbox_area(bbox) <= 0:
            continue
        candidate = dict(block)
        keep = True
        for index, existing in list(enumerate(candidates)):
            overlap = max(bbox_iou(candidate['bbox'], existing['bbox']), bbox_containment(candidate['bbox'], existing['bbox']))
            if overlap < 0.82:
                continue
            cand_score = (block_priority(candidate), float(candidate.get('score') or 0))
            exist_score = (block_priority(existing), float(existing.get('score') or 0))
            if cand_score > exist_score:
                candidates[index] = candidate
            keep = False
            break
        if keep:
            candidates.append(candidate)
    return sorted(candidates, key=lambda b: ((b.get('bbox') or [0, 0, 0, 0])[1], (b.get('bbox') or [0, 0, 0, 0])[0]))


def matching_layout_block(page_blocks, candidate):
    cand_bbox = candidate.get('bbox') or []
    cand_type = candidate.get('category_type') or candidate.get('type')
    best = None
    best_score = -1
    for block in page_blocks:
        if (block.get('category_type') or block.get('type')) != cand_type:
            continue
        bbox = block.get('bbox') or []
        if not bbox:
            continue
        score = max(bbox_iou(cand_bbox, bbox), bbox_containment(cand_bbox, bbox))
        if score > best_score:
            best = block
            best_score = score
    return best if best_score >= 0.82 else None


def normalize_text_crop(crop):
    crop = crop.convert('RGB')
    width, height = crop.size
    if width <= 0 or height <= 0:
        return crop
    scale = max(TEXT_CROP_MIN_WIDTH / max(1, width), TEXT_CROP_MIN_HEIGHT / max(1, height), 1.0)
    scale = min(scale, TEXT_CROP_MAX_UPSCALE)
    if scale > 1.01:
        new_size = (max(1, int(round(width * scale))), max(1, int(round(height * scale))))
        crop = crop.resize(new_size, Image.Resampling.LANCZOS)
        width, height = crop.size
    canvas_w = max(width, TEXT_CROP_MIN_WIDTH)
    canvas_h = max(height, TEXT_CROP_MIN_HEIGHT)
    if canvas_w != width or canvas_h != height:
        canvas = Image.new('RGB', (canvas_w, canvas_h), 'white')
        canvas.paste(crop, ((canvas_w - width) // 2, (canvas_h - height) // 2))
        crop = canvas
    return crop


text_ocr_records = []
for page in page_records:
    page_no = int(page['page_number'])
    page_image_path = OUTPUT_DIR / page['page_image_path']
    image = Image.open(page_image_path).convert('RGB')
    width, height = image.size
    min_text_area = width * height * MIN_TEXT_CROP_AREA_RATIO
    page_blocks = page.get('text_blocks') or []
    text_candidates = [
        candidate for candidate in dedupe_layout_text_candidates(page_blocks)
        if bbox_area(candidate['bbox']) >= min_text_area
    ]
    for idx, candidate in enumerate(text_candidates, start=1):
        layout_type = candidate.get('category_type') or candidate.get('type')
        safe_type = re.sub(r'[^a-z0-9]+', '_', str(layout_type).lower()).strip('_') or 'text'
        block_id = f'page_{page_no:03d}_block_{idx:03d}_{safe_type}'
        x0, y0, x1, y1 = clamp_bbox(candidate['bbox'], width, height, pad=TEXT_CROP_PAD_PX)
        crop_path = TEXT_CROPS_DIR / f'{block_id}.jpg'
        crop = normalize_text_crop(image.crop((x0, y0, x1, y1)))
        crop.save(crop_path, quality=92)
        target_block = matching_layout_block(page_blocks, candidate) or candidate
        target_block.update({
            'ocr_id': block_id,
            'text_crop_path': crop_path.relative_to(OUTPUT_DIR).as_posix(),
            'layout_type': layout_type,
            'ocr_engine': 'deepseek-ocr-block',
        })
        text_ocr_records.append({
            'record_id': block_id,
            'record_type': 'layout_block',
            'block_id': block_id,
            'page_number': page_no,
            'path': str(crop_path.resolve()),
            'result_name': f'{block_id}.json',
            'bbox': target_block.get('bbox'),
            'category_type': layout_type,
            'layout_type': layout_type,
            'order': target_block.get('order'),
            'det_index': target_block.get('det_index'),
        })

TEXT_BLOCK_MANIFEST.write_text('\n'.join(json.dumps(record, ensure_ascii=False) for record in text_ocr_records) + ('\n' if text_ocr_records else ''), encoding='utf-8')
(METADATA_DIR / 'text_ocr_manifest.jsonl').write_text(TEXT_BLOCK_MANIFEST.read_text(encoding='utf-8'), encoding='utf-8')
(METADATA_DIR / 'layout_detections.json').write_text(json.dumps(page_records, ensure_ascii=False, indent=2), encoding='utf-8')
(METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + ('\n' if block_records else ''), encoding='utf-8')
print('text/table blocks for OCR:', len(text_ocr_records))
print('manifest:', TEXT_BLOCK_MANIFEST)


In [ ]:
WORKER_PATH = OUTPUT_DIR / 'deepseek_ocr_worker.py'
worker_py = r"""
import json
import os
import shutil
import sys
import traceback
from pathlib import Path

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from transformers import AutoModel, AutoTokenizer

try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    sys.stderr.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

if not torch.cuda.is_available():
    # DeepSeek-OCR remote code hard-calls `.cuda()` inside infer().
    # CPU mode is unofficial; this monkey patch makes a local benchmark/run possible.
    torch.Tensor.cuda = lambda self, *args, **kwargs: self
    torch.nn.Module.cuda = lambda self, *args, **kwargs: self

manifest_path = Path(sys.argv[1])
out_dir = Path(sys.argv[2])
out_dir.mkdir(parents=True, exist_ok=True)

model_name = os.environ.get('DEEPSEEK_MODEL', 'deepseek-ai/DeepSeek-OCR')
prompt = os.environ.get('DEEPSEEK_PROMPT', '<image>\n<|grounding|>Convert the document to markdown.')
attn_impl = os.environ.get('DEEPSEEK_ATTN_IMPL', 'eager')
base_size = int(os.environ.get('DEEPSEEK_BASE_SIZE', '1024'))
image_size = int(os.environ.get('DEEPSEEK_IMAGE_SIZE', '640'))
crop_mode = os.environ.get('DEEPSEEK_CROP_MODE', '1') == '1'
test_compress = os.environ.get('DEEPSEEK_TEST_COMPRESS', '1') == '1'
save_results = os.environ.get('DEEPSEEK_SAVE_RESULTS', '1') == '1'
cpu_bfloat16 = os.environ.get('DEEPSEEK_CPU_BFLOAT16', '1') == '1'
force = os.environ.get('FORCE_DEEPSEEK', '0') == '1'

print('worker cuda visible =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('torch =', torch.__version__, 'cuda =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f'gpu memory before load: free={free/1024**3:.2f}GB total={total/1024**3:.2f}GB')

records = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
load_kwargs = {
    'trust_remote_code': True,
    'use_safetensors': True,
    '_attn_implementation': attn_impl,
    'low_cpu_mem_usage': True,
}
try:
    model = AutoModel.from_pretrained(model_name, **load_kwargs)
except TypeError:
    load_kwargs.pop('low_cpu_mem_usage', None)
    model = AutoModel.from_pretrained(model_name, **load_kwargs)
except Exception as exc:
    if 'attn' not in str(exc).lower() and 'attention' not in str(exc).lower():
        raise
    load_kwargs.pop('_attn_implementation', None)
    model = AutoModel.from_pretrained(model_name, **load_kwargs)

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if torch.cuda.is_available():
    model = model.eval().cuda().to(dtype)
    free, total = torch.cuda.mem_get_info(0)
    print(f'gpu memory after load: free={free/1024**3:.2f}GB total={total/1024**3:.2f}GB dtype={dtype}')
else:
    model = model.eval()
    if cpu_bfloat16:
        model = model.to(torch.bfloat16)


TEXT_FILE_SUFFIXES = {'.md', '.mmd', '.markdown', '.txt'}


def is_meaningful_text(value):
    text = str(value or '').strip()
    return bool(text) and text.lower() not in {'none', 'null', 'nan'}


def read_saved_markdown(raw_dir):
    candidates = [
        path for path in raw_dir.rglob('*')
        if path.is_file() and path.suffix.lower() in TEXT_FILE_SUFFIXES
    ]
    candidates.sort(key=lambda path: (path.stat().st_mtime, path.stat().st_size), reverse=True)
    for path in candidates:
        try:
            text = path.read_text(encoding='utf-8', errors='ignore').strip()
        except Exception:
            continue
        if is_meaningful_text(text):
            return text, path
    return '', None


def result_to_markdown(result, raw_dir):
    if isinstance(result, str) and is_meaningful_text(result):
        return result, 'return:string'
    if isinstance(result, dict):
        text = result.get('text') or result.get('markdown') or result.get('content')
        if is_meaningful_text(text):
            return str(text), 'return:dict'
        fallback = json.dumps(result, ensure_ascii=False)
        if is_meaningful_text(fallback) and fallback != '{}':
            return fallback, 'return:dict-json'

    saved_text, saved_path = read_saved_markdown(raw_dir)
    if saved_text:
        return saved_text, f'saved:{saved_path.name}'
    if result is not None and is_meaningful_text(result):
        return str(result), 'return:other'
    return '', 'empty'


for record in records:
    page_no = int(record['page_number'])
    image_file = str(record['path'])
    record_type = str(record.get('record_type') or 'page')
    record_id = str(record.get('record_id') or f'page_{page_no:03d}')
    result_name = str(record.get('result_name') or f'{record_id}.json')
    result_path = out_dir / result_name
    if result_path.exists() and not force:
        print('skip existing record', record_id)
        continue
    raw_dir = out_dir / f'raw_{record_id}'
    if raw_dir.exists() and force:
        shutil.rmtree(raw_dir, ignore_errors=True)
    raw_dir.mkdir(parents=True, exist_ok=True)
    try:
        infer_kwargs = dict(
            prompt=prompt,
            image_file=image_file,
            output_path=str(raw_dir),
            base_size=base_size,
            image_size=image_size,
            crop_mode=crop_mode,
            test_compress=test_compress,
        )
        try:
            # Page OCR may need save_results=True to write Markdown files. For tiny block crops,
            # save_results=True can hit DeepSeek's grounding postprocess with zero image tokens.
            result = model.infer(tokenizer, save_results=save_results, **infer_kwargs)
        except TypeError:
            result = model.infer(tokenizer, **infer_kwargs)
        markdown, markdown_source = result_to_markdown(result, raw_dir)
        payload = {
            'record_id': record_id,
            'record_type': record_type,
            'page_number': page_no,
            'page_image_path': image_file,
            'markdown': markdown,
            'markdown_source': markdown_source,
            'raw_output_dir': str(raw_dir),
            'engine': 'deepseek-ocr',
        }
        for key in ['block_id', 'bbox', 'category_type', 'layout_type', 'order', 'det_index']:
            if key in record:
                payload[key] = record.get(key)
        if not is_meaningful_text(markdown):
            payload['error'] = 'DeepSeek returned no markdown text. Check raw_output_dir.'
    except torch.cuda.OutOfMemoryError as exc:
        payload = {
            'record_id': record_id,
            'record_type': record_type,
            'page_number': page_no,
            'page_image_path': image_file,
            'markdown': '',
            'engine': 'deepseek-ocr',
            'error': 'CUDA OutOfMemoryError: ' + str(exc),
            'traceback': traceback.format_exc(),
        }
        result_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        print('OOM on page', page_no)
        raise
    except Exception as exc:
        payload = {
            'record_id': record_id,
            'record_type': record_type,
            'page_number': page_no,
            'page_image_path': image_file,
            'markdown': '',
            'engine': 'deepseek-ocr',
            'error': str(exc),
            'traceback': traceback.format_exc(),
        }
    result_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('done', record_type, record_id, 'chars=', len(payload.get('markdown') or ''), 'error=', payload.get('error'))
    sys.stdout.flush()
"""
WORKER_PATH.write_text(worker_py, encoding='utf-8')
print(WORKER_PATH)


In [ ]:
def selected_gpu_ids():
    if GPU_IDS_TO_USE == 'auto':
        ids = GPU_IDS[:]
    else:
        ids = [int(value.strip()) for value in GPU_IDS_TO_USE.split(',') if value.strip()]
    ids = ids[:max(1, MAX_PARALLEL_GPUS)]
    return ids


def write_jsonl(path, records):
    path.write_text('\n'.join(json.dumps(record, ensure_ascii=False) for record in records) + ('\n' if records else ''), encoding='utf-8')


def run_deepseek_records(records, out_dir, manifest_prefix, prompt, base_size, image_size, crop_mode, test_compress, save_results, force, enabled=True):
    if not enabled:
        print(manifest_prefix, 'disabled, skip.')
        return False
    if not records:
        print(manifest_prefix, 'has no records, skip.')
        return False

    gpu_ids = selected_gpu_ids()
    if not gpu_ids and not ALLOW_CPU_DEEPSEEK:
        raise RuntimeError('No GPU selected for DeepSeek-OCR.')

    worker_count = len(gpu_ids) if gpu_ids else 1
    shards = [[] for _ in range(worker_count)]
    for idx, record in enumerate(records):
        shards[idx % worker_count].append(record)

    processes = []
    for shard_idx, shard in enumerate(shards):
        if not shard:
            continue
        manifest = MANIFEST_DIR / f'{manifest_prefix}_shard_{shard_idx}.jsonl'
        write_jsonl(manifest, shard)
        env = os.environ.copy()
        env['USE_TF'] = '0'
        env['TRANSFORMERS_NO_TF'] = '1'
        env['USE_FLAX'] = '0'
        env['TRANSFORMERS_NO_FLAX'] = '1'
        env['TF_CPP_MIN_LOG_LEVEL'] = '3'
        env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        env['DEEPSEEK_MODEL'] = DEEPSEEK_MODEL
        env['DEEPSEEK_PROMPT'] = prompt
        env['DEEPSEEK_ATTN_IMPL'] = DEEPSEEK_ATTN_IMPL
        env['DEEPSEEK_BASE_SIZE'] = str(base_size)
        env['DEEPSEEK_IMAGE_SIZE'] = str(image_size)
        env['DEEPSEEK_CROP_MODE'] = '1' if crop_mode else '0'
        env['DEEPSEEK_TEST_COMPRESS'] = '1' if test_compress else '0'
        env['DEEPSEEK_SAVE_RESULTS'] = '1' if save_results else '0'
        env['DEEPSEEK_CPU_BFLOAT16'] = '1' if DEEPSEEK_CPU_BFLOAT16 else '0'
        env['FORCE_DEEPSEEK'] = '1' if force else '0'
        gpu_label = 'cpu'
        if gpu_ids:
            gpu_label = str(gpu_ids[shard_idx])
            env['CUDA_VISIBLE_DEVICES'] = gpu_label
        cmd = [sys.executable, str(WORKER_PATH), str(manifest), str(out_dir)]
        print('launch', manifest_prefix, 'shard', shard_idx, 'gpu=', gpu_label, 'records=', len(shard))
        processes.append(subprocess.Popen(cmd, env=env))

    ok = True
    for process in processes:
        code = process.wait()
        ok = ok and (code == 0)
        print(manifest_prefix, 'worker exit =', code)
    return ok


page_ocr_records = [
    {
        'record_id': f'page_{int(page["page_number"]):03d}',
        'record_type': 'page',
        'page_number': int(page['page_number']),
        'path': str(Path(page['path']).resolve()),
        'result_name': f'page_{int(page["page_number"]):03d}.json',
    }
    for page in page_images
]

deepseek_block_ok = run_deepseek_records(
    text_ocr_records,
    BLOCK_OCR_OUT_DIR,
    'deepseek_blocks',
    BLOCK_OCR_PROMPT,
    BLOCK_OCR_BASE_SIZE,
    BLOCK_OCR_IMAGE_SIZE,
    BLOCK_OCR_CROP_MODE,
    BLOCK_OCR_TEST_COMPRESS,
    BLOCK_OCR_SAVE_RESULTS,
    FORCE_DEEPSEEK or FORCE_BLOCK_OCR,
    enabled=RUN_DEEPSEEK_OCR and RUN_BLOCK_OCR,
)
deepseek_ok = run_deepseek_records(
    page_ocr_records,
    DEEPSEEK_OUT_DIR,
    'deepseek_pages',
    DEEPSEEK_PROMPT,
    DEEPSEEK_BASE_SIZE,
    DEEPSEEK_IMAGE_SIZE,
    DEEPSEEK_CROP_MODE,
    DEEPSEEK_TEST_COMPRESS,
    PAGE_OCR_SAVE_RESULTS,
    FORCE_DEEPSEEK or FORCE_PAGE_OCR,
    enabled=RUN_DEEPSEEK_OCR and RUN_PAGE_OCR,
)
print('deepseek_ok =', deepseek_ok)
print('deepseek_block_ok =', deepseek_block_ok)
if RUN_DEEPSEEK_OCR and RUN_BLOCK_OCR and not deepseek_block_ok:
    raise RuntimeError('DeepSeek block OCR failed. Xem traceback ở cell output hoặc JSON trong _deepseek_blocks.')
if RUN_DEEPSEEK_OCR and RUN_PAGE_OCR and not deepseek_ok:
    raise RuntimeError('DeepSeek page OCR failed. Xem traceback ở cell output hoặc page_XXX.json trong _deepseek_pages.')


In [ ]:
import unicodedata

MARKDOWN_IMAGE_LINK_LINE_RE = re.compile(r'^\s*!\[([^\]]*)\]\(([^)]+)\)\s*$')
TEXT_BLOCK_OUTPUT_TYPES = set(TEXT_OCR_TYPES)
VISUAL_OUTPUT_TYPES = {'image', 'table'}


def clean_ocr_block_markdown(value):
    text = str(value or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:markdown|md|text)?\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\s*```$', '', text)
    output_lines = []
    for raw_line in text.splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if not stripped or stripped == '</break>':
            continue
        if re.search(r'(?i)\b(do not|do add|preserve|extract only|return only|convert the|change the|add text|add any text)\b', stripped) and not re.search(r'[À-ỹĐđ]', stripped):
            continue
        if MARKDOWN_IMAGE_LINK_LINE_RE.match(stripped):
            continue
        stripped_no_heading = re.sub(r'^#{1,6}\s*', '', stripped)
        if re.fullmatch(r'PDF Page\s+\d+', stripped_no_heading, flags=re.IGNORECASE):
            continue
        if stripped_no_heading.lower() in {'none', 'null', 'nan'}:
            continue
        output_lines.append(line)
    return '\n'.join(output_lines).strip()


def markdown_text_only(text):
    cleaned = clean_ocr_block_markdown(text)
    lines = []
    for line in cleaned.splitlines():
        stripped = line.strip()
        if not stripped or MARKDOWN_IMAGE_LINK_LINE_RE.match(stripped):
            continue
        stripped = re.sub(r'^#{1,6}\s*', '', stripped)
        lines.append(stripped)
    cleaned = clean_text(' '.join(lines))
    return '' if cleaned.strip().lower() in {'none', 'null', 'nan'} else cleaned


def page_ocr_segments(markdown):
    text = clean_ocr_block_markdown(markdown)
    segments = []
    current = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            if current:
                segments.append('\n'.join(current).strip())
                current = []
            continue
        current.append(raw_line.rstrip())
    if current:
        segments.append('\n'.join(current).strip())
    if not segments and text.strip():
        segments = [text.strip()]
    return [segment for segment in segments if markdown_text_only(segment)]


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        if end < len(text):
            punct = max(text.rfind('.', start, end), text.rfind('?', start, end), text.rfind('!', start, end))
            if punct > start + chunk_size * 0.55:
                end = punct + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]


def load_deepseek_pages():
    pages = {}
    for path in sorted(DEEPSEEK_OUT_DIR.glob('page_*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        pages[int(payload['page_number'])] = payload
    return pages


def load_block_ocr_outputs():
    outputs = {}
    for path in sorted(BLOCK_OCR_OUT_DIR.glob('*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        record_id = payload.get('record_id') or payload.get('block_id') or path.stem
        outputs[str(record_id)] = payload
    return outputs


def pdf_text_for_image_bbox(page_no, image_bbox, rendered_width, rendered_height):
    if not USE_PDF_TEXT_LAYER_FOR_BLOCKS:
        return ''
    try:
        page = pdf_doc[int(page_no) - 1]
        rect = page.rect
        sx = rect.width / max(1, rendered_width)
        sy = rect.height / max(1, rendered_height)
        x0, y0, x1, y1 = [float(value) for value in image_bbox]
        clip = fitz.Rect(x0 * sx, y0 * sy, x1 * sx, y1 * sy)
        return clean_ocr_block_markdown(page.get_text('text', clip=clip) or '')
    except Exception:
        return ''


def fold_text(value):
    text = unicodedata.normalize('NFD', str(value or ''))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return re.sub(r'\s+', ' ', text).lower().strip()


def visual_label_stem(number):
    return re.sub(r'[^0-9]+', '_', number).strip('_')


def find_visual_labels(markdown):
    labels = []
    seen = set()
    pattern = re.compile(r'\b(Hình|Hinh|Bảng|Bang)\s+(\d+(?:[\._-]\d+)+)', re.IGNORECASE)
    for line in str(markdown or '').splitlines():
        for match in pattern.finditer(line):
            kind_raw = fold_text(match.group(1))
            label_type = 'table' if kind_raw.startswith('bang') else 'image'
            stem = visual_label_stem(match.group(2))
            key = (label_type, stem)
            if key in seen:
                continue
            seen.add(key)
            labels.append({
                'type': label_type,
                'label': f'{match.group(1)} {match.group(2)}',
                'stem': stem,
                'line': line.strip(),
                'match_norm': fold_text(match.group(0)),
            })
    return labels


def unique_stem(stem, used):
    candidate = stem
    index = 2
    while candidate in used:
        candidate = f'{stem}_{index}'
        index += 1
    used.add(candidate)
    return candidate


def rename_rel_path(rel_path, new_rel_path):
    if not rel_path:
        return ''
    old_path = OUTPUT_DIR / rel_path
    new_path = OUTPUT_DIR / new_rel_path
    if old_path.exists() and old_path != new_path:
        new_path.parent.mkdir(parents=True, exist_ok=True)
        if not new_path.exists():
            old_path.rename(new_path)
            return new_path.relative_to(OUTPUT_DIR).as_posix()
    return rel_path


def assign_visual_labels(image_records, deepseek_pages):
    used = set()
    for record in image_records:
        if record.get('id'):
            used.add(str(record['id']))

    for page_no, payload in deepseek_pages.items():
        markdown = str(payload.get('markdown') or '')
        labels = find_visual_labels(markdown)
        page_visuals = sorted(
            [record for record in image_records if int(record.get('page_number') or record.get('page') or 0) == int(page_no)],
            key=lambda record: (record.get('bbox') or [0, 0, 0, 0])[1],
        )
        for visual_type in ['image', 'table']:
            visuals = [record for record in page_visuals if record.get('type') == visual_type]
            visual_labels = [label for label in labels if label['type'] == visual_type]
            for record, label in zip(visuals, visual_labels):
                old_id = str(record.get('id') or '')
                used.discard(old_id)
                stem = label['stem'] if visual_type == 'image' else f"bang_{label['stem']}"
                stem = unique_stem(stem, used)
                record['id'] = stem
                record['label'] = label['label']
                record['caption'] = label['line']
                suffix = Path(str(record.get('path') or '')).suffix or '.jpg'
                record['path'] = rename_rel_path(str(record.get('path') or ''), f'images/{stem}{suffix}')
                if record.get('caption_crop_path'):
                    caption_suffix = Path(str(record.get('caption_crop_path'))).suffix or '.jpg'
                    record['caption_crop_path'] = rename_rel_path(str(record.get('caption_crop_path')), f'caption_crops/{stem}_caption{caption_suffix}')


def image_markdown(record):
    alt = str(record.get('caption') or record.get('label') or record.get('id') or 'image').replace('\n', ' ')
    return f'![{alt}]({record.get("path", "")})'


def bbox_values(block):
    bbox = block.get('bbox') or [0, 0, 0, 0]
    return [float(value) for value in bbox]


def reading_order_blocks(blocks, page_width, page_height):
    candidates = [block for block in blocks if block.get('bbox')]
    candidates = sorted(candidates, key=lambda block: (bbox_values(block)[1], bbox_values(block)[0]))
    bands = []
    for block in candidates:
        x0, y0, x1, y1 = bbox_values(block)
        height = max(1.0, y1 - y0)
        placed = False
        for band in bands:
            overlap = min(y1, band['y1']) - max(y0, band['y0'])
            tolerance = max(18.0, min(height, band['height']) * 0.45)
            if overlap >= -tolerance:
                band['blocks'].append(block)
                band['y0'] = min(band['y0'], y0)
                band['y1'] = max(band['y1'], y1)
                band['height'] = max(1.0, band['y1'] - band['y0'])
                placed = True
                break
        if not placed:
            bands.append({'y0': y0, 'y1': y1, 'height': height, 'blocks': [block]})
    ordered = []
    for band in sorted(bands, key=lambda item: item['y0']):
        ordered.extend(sorted(band['blocks'], key=lambda block: (bbox_values(block)[0], bbox_values(block)[1])))
    return ordered


def attach_block_ocr_text(page_records, block_ocr_outputs):
    block_errors = []
    block_text_count = 0
    for page in page_records:
        page_no = int(page.get('page_number') or 0)
        rendered_width = page.get('width') or 1
        rendered_height = page.get('height') or 1
        for block in page.get('text_blocks') or []:
            ocr_id = block.get('ocr_id')
            if not ocr_id:
                continue
            text_layer_text = pdf_text_for_image_bbox(page_no, block.get('bbox') or [0, 0, 0, 0], rendered_width, rendered_height)
            if text_layer_text:
                block['text'] = text_layer_text
                block['ocr_chars'] = len(markdown_text_only(text_layer_text))
                block['ocr_source'] = 'pdf_text_layer_clip'
                block_text_count += 1
                continue
            payload = block_ocr_outputs.get(str(ocr_id))
            if not payload:
                if RUN_BLOCK_OCR:
                    block['ocr_error'] = 'missing block OCR output'
                    block_errors.append({'block_id': ocr_id, 'page_number': page.get('page_number'), 'error': block['ocr_error']})
                continue
            text = clean_ocr_block_markdown(payload.get('markdown') or '')
            block['text'] = text
            block['ocr_markdown'] = payload.get('markdown') or ''
            block['ocr_chars'] = len(markdown_text_only(text))
            block['ocr_source'] = payload.get('markdown_source')
            if payload.get('error'):
                block['ocr_error'] = payload.get('error')
                block_errors.append({'block_id': ocr_id, 'page_number': page.get('page_number'), 'error': payload.get('error')})
            if len(markdown_text_only(text)) >= MIN_BLOCK_TEXT_CHARS:
                block_text_count += 1
    return block_text_count, block_errors


def assign_page_ocr_to_empty_layout_blocks(page_records, deepseek_pages):
    if not USE_PAGE_OCR_TO_LAYOUT_BLOCKS:
        return 0
    assigned = 0
    for page in page_records:
        page_no = int(page.get('page_number') or 0)
        segments = page_ocr_segments((deepseek_pages.get(page_no) or {}).get('markdown') or '')
        if not segments:
            continue
        text_blocks = []
        for block in reading_order_blocks(page.get('text_blocks') or [], page.get('width') or 1, page.get('height') or 1):
            layout_type = block.get('layout_type') or block.get('category_type') or block.get('type')
            if layout_type in TEXT_BLOCK_OUTPUT_TYPES:
                text_blocks.append(block)
        segment_index = 0
        for block in text_blocks:
            existing = markdown_text_only(block.get('text') or '')
            if existing:
                continue
            if segment_index >= len(segments):
                break
            block['text'] = segments[segment_index]
            block['ocr_chars'] = len(markdown_text_only(segments[segment_index]))
            block['ocr_source'] = 'deepseek-ocr-page-assigned'
            assigned += 1
            segment_index += 1
        if segment_index < len(segments):
            leftovers = [segment for segment in segments[segment_index:] if markdown_text_only(segment)]
            if leftovers:
                page.setdefault('page_ocr_leftover_text', '\n\n'.join(leftovers))
    return assigned


def page_label_sources(page_no, deepseek_pages):
    texts = []
    for page in page_records:
        if int(page.get('page_number')) != int(page_no):
            continue
        ordered = reading_order_blocks(page.get('text_blocks') or [], page.get('width') or 1, page.get('height') or 1)
        for block in ordered:
            layout_type = block.get('layout_type') or block.get('category_type') or block.get('type')
            if layout_type in {'figure_caption', 'table_caption'} and block.get('text'):
                texts.append(str(block.get('text')))
    payload = deepseek_pages.get(int(page_no), {})
    if payload.get('markdown'):
        texts.append(str(payload.get('markdown')))
    return '\n'.join(texts)


def assign_visual_labels_from_layout_text(image_records, deepseek_pages):
    pages = sorted({int(record.get('page_number') or record.get('page') or 0) for record in image_records if record.get('page_number') or record.get('page')})
    label_pages = {
        page_no: {'markdown': page_label_sources(page_no, deepseek_pages)}
        for page_no in pages
    }
    assign_visual_labels(image_records, label_pages)


def block_to_output_items(block):
    block_type = block.get('type')
    layout_type = block.get('layout_type') or block.get('category_type') or block_type
    if block_type in VISUAL_OUTPUT_TYPES and block.get('path'):
        items = [dict(block)]
        table_text = clean_ocr_block_markdown(block.get('text') or '') if layout_type == 'table' else ''
        if table_text:
            items.append({
                'type': 'text',
                'layout_type': 'table_ocr_text',
                'text': table_text,
                'bbox': block.get('bbox'),
                'source': 'deepseek-ocr-block',
            })
        return items
    if layout_type in TEXT_BLOCK_OUTPUT_TYPES:
        text = clean_ocr_block_markdown(block.get('text') or '')
        if not text:
            return []
        return [{
            'type': 'text',
            'layout_type': layout_type,
            'text': text,
            'bbox': block.get('bbox'),
            'score': block.get('score'),
            'ocr_id': block.get('ocr_id'),
            'text_crop_path': block.get('text_crop_path'),
            'source': 'deepseek-ocr-block',
        }]
    return []


def page_output_blocks(layout_page, deepseek_payload):
    raw_blocks = []
    for block in layout_page.get('text_blocks') or []:
        raw_blocks.extend(block_to_output_items(block))
    ordered_blocks = reading_order_blocks(raw_blocks, layout_page.get('width') or 1, layout_page.get('height') or 1)
    leftover_text = clean_ocr_block_markdown(layout_page.get('page_ocr_leftover_text') or '')
    if leftover_text:
        ordered_blocks.append({
            'type': 'text',
            'layout_type': 'page_ocr_leftover',
            'text': leftover_text,
            'bbox': [0, layout_page.get('height') or 0, layout_page.get('width') or 0, layout_page.get('height') or 0],
            'source': 'deepseek-ocr-page-leftover',
        })
    has_text = any(block.get('type') == 'text' and markdown_text_only(block.get('text') or '') for block in ordered_blocks)
    if (not ordered_blocks or not has_text) and deepseek_payload.get('markdown'):
        fallback_text = clean_ocr_block_markdown(deepseek_payload.get('markdown'))
        if fallback_text:
            fallback_block = {
                'type': 'text',
                'layout_type': 'page_fallback',
                'text': fallback_text,
                'bbox': [0, 0, layout_page.get('width') or 0, layout_page.get('height') or 0],
                'source': 'deepseek-ocr-page-fallback',
            }
            if ordered_blocks:
                ordered_blocks = [fallback_block] + ordered_blocks
            else:
                ordered_blocks = [fallback_block]
    return ordered_blocks


def markdown_for_output_block(block):
    if block.get('type') in VISUAL_OUTPUT_TYPES and block.get('path'):
        return image_markdown(block)
    text = clean_ocr_block_markdown(block.get('text') or '')
    if not text:
        return ''
    layout_type = block.get('layout_type')
    if layout_type == 'title' and not text.lstrip().startswith('#'):
        return '### ' + text.replace('\n', ' ')
    return text


layout_page_records = list(page_records)
layout_pages_by_no = {int(page['page_number']): page for page in layout_page_records}
deepseek_pages = load_deepseek_pages() if RUN_DEEPSEEK_OCR else {}
block_ocr_outputs = load_block_ocr_outputs() if RUN_DEEPSEEK_OCR and RUN_BLOCK_OCR else {}
block_text_count, block_ocr_errors = attach_block_ocr_text(page_records, block_ocr_outputs)
page_ocr_assigned_blocks = assign_page_ocr_to_empty_layout_blocks(page_records, deepseek_pages)
block_text_count = sum(
    1
    for page in page_records
    for block in page.get('text_blocks') or []
    if block.get('ocr_id') and len(markdown_text_only(block.get('text') or '')) >= MIN_BLOCK_TEXT_CHARS
)
assign_visual_labels_from_layout_text(image_records, deepseek_pages)

book_lines = [
    f'# {INPUT_PDF.stem}',
    '',
    f'> Source PDF: `{INPUT_PDF}`',
    f'> Generated at: `{datetime.now(timezone.utc).isoformat()}`',
    '> Layout engine: `PDF-Extract-Kit DocLayout-YOLO`',
    f'> OCR engine: `{"DeepSeek-OCR block-level layout-first" if block_ocr_outputs else "DeepSeek-OCR page fallback" if deepseek_pages else "PDF text layer / none"}`',
    '',
]
page_records = []
block_records = []
rag_records = []
errors = list(block_ocr_errors)

for page in page_images:
    page_no = int(page['page_number'])
    layout_page = layout_pages_by_no.get(page_no, {})
    payload = deepseek_pages.get(page_no, {})
    if payload.get('error'):
        errors.append({'page_number': page_no, 'error': payload.get('error')})

    page_blocks = page_output_blocks(layout_page, payload)
    page_lines = [markdown_for_output_block(block) for block in page_blocks]
    page_lines = [line for line in page_lines if str(line).strip()]
    if not page_lines:
        page_lines = [f'[OCR missing for PDF page {page_no}]']
    book_lines += [f'## PDF Page {page_no}', '', *page_lines, '', '</break>', '']

    text = markdown_text_only('\n\n'.join(page_lines))
    image_refs = [
        {key: block.get(key) for key in ['id', 'path', 'caption_crop_path', 'label', 'caption', 'type']}
        for block in page_blocks
        if block.get('type') in {'image', 'table'}
    ]
    page_record = {
        'source_pdf': str(INPUT_PDF),
        'page_number': page_no,
        'page_image_path': layout_page.get('page_image_path') or str(page.get('path')),
        'width': layout_page.get('width'),
        'height': layout_page.get('height'),
        'text': text,
        'markdown': '\n\n'.join(page_lines),
        'layout_dets': layout_page.get('layout_dets') or [],
        'text_blocks': page_blocks,
        'images': image_refs,
        'engine': {'layout': 'pdf_extract_kit_doclayout_yolo', 'ocr': 'deepseek-ocr-block' if block_ocr_outputs else 'deepseek-ocr-page-fallback' if payload else 'pdf_text_layer_or_none'},
        'ocr_error': payload.get('error'),
    }
    page_records.append(page_record)
    for block in page_blocks:
        block_record = dict(block)
        block_record['page_number'] = page_no
        block_record['order'] = len([b for b in block_records if int(b.get('page_number') or -1) == page_no])
        block_records.append(block_record)
    (METADATA_DIR / 'pages').mkdir(parents=True, exist_ok=True)
    (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')

    for idx, chunk in enumerate(chunk_text(text), start=1):
        rag_records.append({
            'chunk_id': f'page_{page_no:03d}_{idx:02d}',
            'source_pdf': str(INPUT_PDF),
            'page_number': page_no,
            'text': chunk,
            'images': image_refs,
            'engine': 'pdf_extract_kit_doclayout_yolo+deepseek_ocr',
        })
    if not text and image_refs:
        rag_records.append({
            'chunk_id': f'page_{page_no:03d}_layout_01',
            'source_pdf': str(INPUT_PDF),
            'page_number': page_no,
            'text': '',
            'images': image_refs,
            'engine': 'pdf_extract_kit_doclayout_yolo+deepseek_ocr',
        })

(OUTPUT_DIR / 'book.md').write_text('\n'.join(book_lines).strip() + '\n', encoding='utf-8')
(OUTPUT_DIR / 'rag_chunks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rag_records) + ('\n' if rag_records else ''), encoding='utf-8')
(METADATA_DIR / 'images.json').write_text(json.dumps(image_records, ensure_ascii=False, indent=2), encoding='utf-8')
(METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + ('\n' if block_records else ''), encoding='utf-8')

summary = {
    'source_pdf': str(INPUT_PDF),
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'page_range': {'start': START_PAGE, 'end': END_PAGE},
    'pages_processed': len(page_records),
    'engine': {'layout': 'pdf_extract_kit_doclayout_yolo', 'ocr': 'deepseek-ocr-block' if block_ocr_outputs else 'deepseek-ocr-page-fallback' if deepseek_pages else 'pdf_text_layer_or_none'},
    'layout_model': {'weight': str(weight_path), 'img_size': IMG_SIZE, 'conf_thres': CONF_THRES, 'iou_thres': IOU_THRES, 'device': LAYOUT_DEVICE},
    'deepseek': {
        'enabled': RUN_DEEPSEEK_OCR,
        'page_ok': bool(globals().get('deepseek_ok', False)),
        'block_ok': bool(globals().get('deepseek_block_ok', False)),
        'model': DEEPSEEK_MODEL,
        'page_prompt': DEEPSEEK_PROMPT,
        'block_prompt': BLOCK_OCR_PROMPT,
        'page_base_size': DEEPSEEK_BASE_SIZE,
        'block_base_size': BLOCK_OCR_BASE_SIZE,
        'image_size': DEEPSEEK_IMAGE_SIZE,
        'max_parallel_gpus': MAX_PARALLEL_GPUS,
    },
    'stats': {
        'blocks': len(block_records),
        'images': len([r for r in image_records if r.get('type') == 'image']),
        'tables': len([r for r in image_records if r.get('type') == 'table']),
        'rag_chunks': len(rag_records),
        'ocr_pages': len(deepseek_pages),
        'ocr_blocks_expected': len(text_ocr_records),
        'ocr_blocks': len(block_ocr_outputs),
        'ocr_blocks_with_text': block_text_count,
        'page_ocr_assigned_blocks': page_ocr_assigned_blocks,
        'ocr_block_errors': len(block_ocr_errors),
        'ocr_errors': len(errors),
    },
    'errors': errors[:20],
    'outputs': {'markdown': 'book.md', 'rag_chunks': 'rag_chunks.jsonl', 'page_metadata_dir': 'metadata/pages', 'block_metadata': 'metadata/blocks.jsonl', 'image_metadata': 'metadata/images.json', 'layout_metadata': 'metadata/layout_detections.json', 'text_ocr_manifest': 'metadata/text_ocr_manifest.jsonl'},
}
(METADATA_DIR / 'book.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Optional Phase 1 post-OCR correction bằng LLM local, không gọi API.
# Giữ lại book_raw_ocr.md để audit, rồi ghi bản đã sửa vào book_corrected.md và book.md.
import shutil

PAGE_HEADING_RE = re.compile(r'(?m)^## PDF Page\s+(\d+)\s*$')
IMAGE_TOKEN_RE = re.compile(r'^!\[.*?\]\(.*?\)\s*$')
IMAGE_LINK_RE = re.compile(r'!\[(.*?)\]\((.*?)\)')


def split_page_sections(markdown):
    matches = list(PAGE_HEADING_RE.finditer(markdown))
    if not matches:
        return markdown, []
    preamble = markdown[:matches[0].start()].rstrip() + '\n\n'
    sections = []
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
        sections.append({'section_index': index, 'page_number': int(match.group(1)), 'text': markdown[match.start():end].strip()})
    return preamble, sections


def protect_markdown_lines(text):
    placeholders = {}
    protected_lines = []
    for line in str(text).splitlines():
        if IMAGE_TOKEN_RE.match(line.strip()):
            token = f'@@IMAGE_LINK_{len(placeholders):04d}@@'
            placeholders[token] = line
            protected_lines.append(token)
        else:
            protected_lines.append(line)
    return '\n'.join(protected_lines), placeholders


def restore_markdown_lines(text, placeholders):
    for token, original in placeholders.items():
        text = str(text).replace(token, original)
    return str(text)


def page_visual_records(page_no):
    return sorted(
        [
            record for record in image_records
            if int(record.get('page_number') or record.get('page') or 0) == int(page_no) and record.get('path')
        ],
        key=lambda record: (record.get('bbox') or [0, 0, 0, 0])[1],
    )


def safe_image_markdown(record):
    if 'image_markdown' in globals():
        return image_markdown(record)
    alt = str(record.get('caption') or record.get('label') or record.get('id') or 'image').replace('\n', ' ')
    return f'![{alt}]({record.get("path", "")})'


def sanitize_and_complete_section(text, page_no):
    valid_paths = {str(record.get('path')) for record in image_records if record.get('path')}
    output_lines = []
    existing_paths = set()
    saw_break = False
    page_visuals = page_visual_records(page_no)

    def next_missing_visual():
        for record in page_visuals:
            rel_path = str(record.get('path') or '')
            if rel_path and rel_path not in existing_paths:
                return record
        return None

    def emit_missing_visual():
        record = next_missing_visual()
        if not record:
            return False
        rel_path = str(record.get('path') or '')
        output_lines.append(safe_image_markdown(record))
        existing_paths.add(rel_path)
        return True

    def replace_inline_image(match):
        rel_path = match.group(2)
        if rel_path in valid_paths:
            existing_paths.add(rel_path)
            return match.group(0)
        return ''

    for raw_line in str(text).splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if not stripped:
            output_lines.append(line)
            continue
        if stripped == '</break>':
            saw_break = True
            continue
        image_match = IMAGE_LINK_RE.fullmatch(stripped)
        if image_match:
            rel_path = image_match.group(2)
            if rel_path in valid_paths and rel_path not in existing_paths:
                output_lines.append(stripped)
                existing_paths.add(rel_path)
            elif rel_path not in valid_paths:
                emit_missing_visual()
            continue
        if 'IMAGE_LINK_' in stripped:
            emit_missing_visual()
            continue
        cleaned_line = IMAGE_LINK_RE.sub(replace_inline_image, line).rstrip()
        if 'IMAGE_LINK_' in cleaned_line:
            emit_missing_visual()
            continue
        if cleaned_line.strip():
            output_lines.append(cleaned_line)

    for record in page_visuals:
        rel_path = str(record.get('path') or '')
        if rel_path and rel_path not in existing_paths:
            output_lines.append(safe_image_markdown(record))
            existing_paths.add(rel_path)
    if saw_break or not any(line.strip() == '</break>' for line in output_lines):
        output_lines.append('</break>')
    return '\n'.join(output_lines).strip()


SPELLCHECK_WORKER_PATH = OUTPUT_DIR / 'spellcheck_worker.py'
spellcheck_worker_py = r"""
import json
import os
import re
import sys
import traceback
from pathlib import Path

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

PLACEHOLDER_RE = re.compile(r'@@IMAGE_LINK_\d{4}@@')


def strip_code_fence(text):
    text = str(text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:markdown|md|text)?\s*', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\s*```$', '', text)
    return text.strip()


def split_long_text(text, max_chars):
    text = str(text or '')
    if len(text) <= max_chars:
        return [text]
    parts = []
    current = []
    current_len = 0
    for paragraph in re.split(r'(\n\s*\n)', text):
        if current_len + len(paragraph) > max_chars and current:
            parts.append(''.join(current).strip())
            current = []
            current_len = 0
        current.append(paragraph)
        current_len += len(paragraph)
    if current:
        parts.append(''.join(current).strip())
    return [part for part in parts if part]


def load_model(model_name):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    kwargs = {'trust_remote_code': True, 'torch_dtype': dtype, 'device_map': 'auto'}
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation='sdpa', **kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()
    return tokenizer, model


def generate_text(tokenizer, model, protected_markdown, max_new_tokens):
    import torch

    messages = [
        {
            'role': 'system',
            'content': (
                'Bạn là bộ hậu xử lý OCR tiếng Việt cho sách giáo khoa. '
                'Chỉ sửa lỗi OCR, chính tả, dấu tiếng Việt, ký tự rác và lỗi tách từ. '
                'Không thêm kiến thức mới, không diễn giải, không tóm tắt, không đổi số liệu hoặc tên riêng nếu không chắc.'
            ),
        },
        {
            'role': 'user',
            'content': (
                'Sửa đoạn Markdown OCR dưới đây.\n'
                'Yêu cầu bắt buộc:\n'
                '- Giữ nguyên heading Markdown, số thứ tự câu hỏi, `</break>` và placeholder dạng @@IMAGE_LINK_0000@@.\n'
                '- Không xóa hoặc thêm ảnh, không đổi đường dẫn ảnh.\n'
                '- Output duy nhất là Markdown đã sửa, không giải thích.\n\n'
                f'```markdown\n{protected_markdown}\n```'
            ),
        },
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[-1]:]
    return strip_code_fence(tokenizer.decode(generated, skip_special_tokens=True))


def correct_text(tokenizer, model, text, max_section_chars, max_new_tokens):
    corrected_parts = []
    for part in split_long_text(text, max_section_chars):
        corrected_parts.append(generate_text(tokenizer, model, part, max_new_tokens))
    return '\n\n'.join(corrected_parts).strip()


manifest_path = Path(sys.argv[1])
out_dir = Path(sys.argv[2])
out_dir.mkdir(parents=True, exist_ok=True)
model_name = os.environ.get('SPELLCHECK_MODEL', 'Qwen/Qwen2.5-3B-Instruct')
max_section_chars = int(os.environ.get('SPELLCHECK_MAX_SECTION_CHARS', '4200'))
max_new_tokens = int(os.environ.get('SPELLCHECK_MAX_NEW_TOKENS', '4096'))
force = os.environ.get('FORCE_SPELLCHECK', '0') == '1'

print('spellcheck cuda visible =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('spellcheck model =', model_name)
records = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
tokenizer, model = load_model(model_name)

for record in records:
    section_index = int(record['section_index'])
    page_number = int(record['page_number'])
    out_path = out_dir / f'section_{section_index:04d}.json'
    if out_path.exists() and not force:
        print('skip existing spellcheck section', section_index)
        continue
    protected_text = str(record['protected_text'])
    expected_placeholders = PLACEHOLDER_RE.findall(protected_text)
    try:
        corrected = correct_text(tokenizer, model, protected_text, max_section_chars, max_new_tokens)
        missing = [token for token in expected_placeholders if token not in corrected]
        if missing or f'## PDF Page {page_number}' not in corrected:
            payload = {
                'section_index': section_index,
                'page_number': page_number,
                'corrected_protected_text': protected_text,
                'error': 'LLM output failed structure guard; kept raw section.',
                'missing_placeholders': missing,
            }
        else:
            payload = {
                'section_index': section_index,
                'page_number': page_number,
                'corrected_protected_text': corrected,
            }
    except Exception as exc:
        payload = {
            'section_index': section_index,
            'page_number': page_number,
            'corrected_protected_text': protected_text,
            'error': str(exc),
            'traceback': traceback.format_exc(),
        }
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('spellcheck page', page_number, 'error=', payload.get('error'))
    sys.stdout.flush()
"""
SPELLCHECK_WORKER_PATH.write_text(spellcheck_worker_py, encoding='utf-8')


def selected_spellcheck_gpu_ids():
    if SPELLCHECK_GPU_IDS_TO_USE == 'auto':
        ids = [str(gpu_id) for gpu_id in GPU_IDS]
    else:
        ids = [value.strip() for value in SPELLCHECK_GPU_IDS_TO_USE.split(',') if value.strip()]
    return ids[:max(1, SPELLCHECK_MAX_PARALLEL_GPUS)]


def run_spellcheck_workers(records):
    out_dir = OUTPUT_DIR / '_spellcheck_pages'
    out_dir.mkdir(parents=True, exist_ok=True)
    gpu_ids = selected_spellcheck_gpu_ids()
    if not gpu_ids and not ALLOW_CPU_DEEPSEEK:
        raise RuntimeError('No GPU selected for spellcheck LLM.')

    worker_count = len(gpu_ids) if gpu_ids else 1
    shards = [[] for _ in range(worker_count)]
    for idx, record in enumerate(records):
        shards[idx % worker_count].append(record)

    processes = []
    for shard_idx, shard in enumerate(shards):
        if not shard:
            continue
        manifest = MANIFEST_DIR / f'spellcheck_shard_{shard_idx}.jsonl'
        write_jsonl(manifest, shard)
        env = os.environ.copy()
        env['USE_TF'] = '0'
        env['TRANSFORMERS_NO_TF'] = '1'
        env['USE_FLAX'] = '0'
        env['TRANSFORMERS_NO_FLAX'] = '1'
        env['TF_CPP_MIN_LOG_LEVEL'] = '3'
        env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        env['SPELLCHECK_MODEL'] = SPELLCHECK_MODEL
        env['SPELLCHECK_MAX_SECTION_CHARS'] = str(SPELLCHECK_MAX_SECTION_CHARS)
        env['SPELLCHECK_MAX_NEW_TOKENS'] = str(SPELLCHECK_MAX_NEW_TOKENS)
        env['FORCE_SPELLCHECK'] = '1' if FORCE_SPELLCHECK else '0'
        gpu_label = 'cpu'
        if gpu_ids:
            gpu_label = gpu_ids[shard_idx]
            env['CUDA_VISIBLE_DEVICES'] = gpu_label
        cmd = [sys.executable, str(SPELLCHECK_WORKER_PATH), str(manifest), str(out_dir)]
        print('launch spellcheck shard', shard_idx, 'gpu=', gpu_label, 'sections=', len(shard))
        processes.append(subprocess.Popen(cmd, env=env))

    ok = True
    for process in processes:
        code = process.wait()
        ok = ok and (code == 0)
        print('spellcheck worker exit =', code)
    return ok, out_dir


def section_blocks(section_text, page_no, image_records_by_path):
    blocks = []
    order = 0
    for raw_line in str(section_text).splitlines():
        line = raw_line.strip()
        if not line or line == '</break>':
            continue
        image_match = IMAGE_LINK_RE.fullmatch(line)
        if image_match:
            rel_path = image_match.group(2)
            source_record = image_records_by_path.get(rel_path)
            if not source_record:
                continue
            record = dict(source_record)
            record['order'] = order
            record['page'] = page_no
            record['page_number'] = page_no
            blocks.append(record)
        else:
            blocks.append({'type': 'text', 'order': order, 'text': raw_line.rstrip(), 'page': page_no, 'page_number': page_no, 'source': 'spellcheck' if RUN_LLM_SPELLCHECK else 'deepseek-ocr'})
        order += 1
    return blocks


def rebuild_outputs_from_book(markdown_path, spellcheck_meta):
    text = markdown_path.read_text(encoding='utf-8')
    _, sections = split_page_sections(text)
    image_records_by_path = {str(record.get('path')): record for record in image_records if record.get('path')}
    layout_pages_by_no = {int(page.get('page_number')): page for page in page_records}
    rebuilt_pages = []
    rebuilt_blocks = []
    rebuilt_rag = []

    for section in sections:
        page_no = int(section['page_number'])
        layout_page = layout_pages_by_no.get(page_no, {})
        blocks = section_blocks(section['text'], page_no, image_records_by_path)
        page_text = markdown_text_only(section['text'])
        image_refs = [
            {key: block.get(key) for key in ['id', 'path', 'caption_crop_path', 'label', 'caption', 'type']}
            for block in blocks
            if block.get('type') in {'image', 'table'}
        ]
        page_record = {
            'source_pdf': str(INPUT_PDF),
            'page_number': page_no,
            'page_image_path': layout_page.get('page_image_path'),
            'width': layout_page.get('width'),
            'height': layout_page.get('height'),
            'text': page_text,
            'markdown': section['text'],
            'layout_dets': layout_page.get('layout_dets') or [],
            'text_blocks': blocks,
            'images': image_refs,
            'engine': {'layout': 'pdf_extract_kit_doclayout_yolo', 'ocr': 'deepseek-ocr', 'post_ocr': 'qwen_spellcheck' if spellcheck_meta.get('applied') else 'none'},
        }
        rebuilt_pages.append(page_record)
        (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')
        for block in blocks:
            block_record = dict(block)
            block_record['page_number'] = page_no
            rebuilt_blocks.append(block_record)
        for idx, chunk in enumerate(chunk_text(page_text), start=1):
            rebuilt_rag.append({
                'chunk_id': f'page_{page_no:03d}_{idx:02d}',
                'source_pdf': str(INPUT_PDF),
                'page_number': page_no,
                'text': chunk,
                'images': image_refs,
                'engine': 'pdf_extract_kit_doclayout_yolo+deepseek_ocr+qwen_spellcheck' if spellcheck_meta.get('applied') else 'pdf_extract_kit_doclayout_yolo+deepseek_ocr',
            })

    (METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rebuilt_blocks) + ('\n' if rebuilt_blocks else ''), encoding='utf-8')
    (OUTPUT_DIR / 'rag_chunks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rebuilt_rag) + ('\n' if rebuilt_rag else ''), encoding='utf-8')
    summary_path = METADATA_DIR / 'book.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
    summary['spellcheck'] = spellcheck_meta
    summary.setdefault('stats', {})
    summary['stats']['blocks'] = len(rebuilt_blocks)
    summary['stats']['rag_chunks'] = len(rebuilt_rag)
    summary['stats']['pages_after_rebuild'] = len(rebuilt_pages)
    summary.setdefault('outputs', {})
    summary['outputs']['raw_markdown'] = 'book_raw_ocr.md'
    summary['outputs']['corrected_markdown'] = 'book_corrected.md'
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return rebuilt_pages, rebuilt_blocks, rebuilt_rag


spellcheck_meta = {'enabled': RUN_LLM_SPELLCHECK, 'applied': False, 'model': SPELLCHECK_MODEL if RUN_LLM_SPELLCHECK else None, 'errors': []}
if RUN_LLM_SPELLCHECK:
    book_md_path = OUTPUT_DIR / 'book.md'
    raw_md_path = OUTPUT_DIR / 'book_raw_ocr.md'
    corrected_md_path = OUTPUT_DIR / 'book_corrected.md'
    shutil.copy2(book_md_path, raw_md_path)
    preamble, sections = split_page_sections(raw_md_path.read_text(encoding='utf-8'))
    protected_records = []
    placeholders_by_section = {}
    for section in sections:
        protected, placeholders = protect_markdown_lines(section['text'])
        placeholders_by_section[int(section['section_index'])] = placeholders
        protected_records.append({**section, 'protected_text': protected})

    ok, spellcheck_out_dir = run_spellcheck_workers(protected_records)
    if not ok and SPELLCHECK_STRICT:
        raise RuntimeError('Spellcheck worker failed.')
    if not ok:
        spellcheck_meta['errors'].append({'stage': 'worker', 'error': 'worker exited non-zero; kept raw OCR markdown'})
    else:
        corrected_sections = []
        for record in protected_records:
            section_index = int(record['section_index'])
            page_number = int(record['page_number'])
            result_path = spellcheck_out_dir / f'section_{section_index:04d}.json'
            if result_path.exists():
                payload = json.loads(result_path.read_text(encoding='utf-8'))
                protected_text = str(payload.get('corrected_protected_text') or record['protected_text'])
                if payload.get('error'):
                    spellcheck_meta['errors'].append({'page_number': page_number, 'error': payload.get('error')})
            else:
                protected_text = str(record['protected_text'])
                spellcheck_meta['errors'].append({'page_number': page_number, 'error': 'missing spellcheck output'})
            restored = restore_markdown_lines(protected_text, placeholders_by_section.get(section_index, {}))
            if f'## PDF Page {page_number}' not in restored:
                restored = record['text']
                spellcheck_meta['errors'].append({'page_number': page_number, 'error': 'missing page heading after restore; kept raw section'})
            restored = sanitize_and_complete_section(restored, page_number)
            corrected_sections.append(restored.strip())

        corrected_markdown = (preamble + '\n\n'.join(corrected_sections)).strip() + '\n'
        corrected_md_path.write_text(corrected_markdown, encoding='utf-8')
        if SPELLCHECK_OVERWRITE_BOOK_MD:
            book_md_path.write_text(corrected_markdown, encoding='utf-8')
            spellcheck_meta['applied'] = True
    spellcheck_meta['error_count'] = len(spellcheck_meta['errors'])
else:
    print('RUN_LLM_SPELLCHECK=False, skip post-OCR correction.')

if spellcheck_meta.get('applied'):
    page_records, block_records, rag_records = rebuild_outputs_from_book(OUTPUT_DIR / 'book.md', spellcheck_meta)
    print('rebuilt pages =', len(page_records), 'rag chunks =', len(rag_records))
else:
    summary_path = METADATA_DIR / 'book.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        summary['spellcheck'] = spellcheck_meta
        summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('spellcheck applied =', spellcheck_meta.get('applied'), 'errors =', spellcheck_meta.get('error_count', 0))


In [ ]:
# Phase 1 quality audit: không thay đổi dữ liệu, chỉ ghi report để kiểm tra nhanh.
def find_image_links(markdown):
    return IMAGE_LINK_RE.findall(markdown)


def page_text_lengths(markdown):
    _, sections = split_page_sections(markdown)
    lengths = []
    for section in sections:
        lengths.append({'page_number': int(section['page_number']), 'chars': len(markdown_text_only(section['text']))})
    return lengths


def duplicate_values(values):
    seen = set()
    dupes = set()
    for value in values:
        if value in seen:
            dupes.add(value)
        seen.add(value)
    return sorted(dupes)


book_md = OUTPUT_DIR / 'book.md'
markdown = book_md.read_text(encoding='utf-8') if book_md.exists() else ''
image_links = find_image_links(markdown)
broken_links = []
for _, rel_path in image_links:
    if not (OUTPUT_DIR / rel_path).exists():
        broken_links.append(rel_path)

image_ids = [str(record.get('id')) for record in image_records if record.get('id')]
lengths = page_text_lengths(markdown)
empty_pages = [item['page_number'] for item in lengths if item['chars'] == 0]
short_pages = [item for item in lengths if 0 < item['chars'] < MIN_PAGE_TEXT_CHARS]
none_like_pages = []
for section in split_page_sections(markdown)[1]:
    text_only = markdown_text_only(section['text']).strip().lower()
    if text_only in {'none', 'null', 'nan'}:
        none_like_pages.append(int(section['page_number']))
empty_page_ratio = len(empty_pages) / max(1, len(lengths))
short_page_ratio = len(short_pages) / max(1, len(lengths))
deepseek_errors = []
for path in sorted(DEEPSEEK_OUT_DIR.glob('page_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    if payload.get('error'):
        deepseek_errors.append({'page_number': payload.get('page_number'), 'error': payload.get('error')})

rag_path = OUTPUT_DIR / 'rag_chunks.jsonl'
rag_count = len([line for line in rag_path.read_text(encoding='utf-8').splitlines() if line.strip()]) if rag_path.exists() else 0
block_ocr_expected = len(text_ocr_records) if 'text_ocr_records' in globals() else 0
block_ocr_outputs_count = len(block_ocr_outputs) if 'block_ocr_outputs' in globals() else 0
block_ocr_with_text = block_text_count if 'block_text_count' in globals() else 0
block_ocr_text_coverage = block_ocr_with_text / max(1, block_ocr_expected)
quality_report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'pages_expected': len(page_images),
    'pages_in_markdown': len(lengths),
    'image_records': len(image_records),
    'image_links_in_markdown': len(image_links),
    'broken_image_links': broken_links,
    'duplicate_image_ids': duplicate_values(image_ids),
    'empty_pages': empty_pages,
    'short_pages': short_pages,
    'none_like_pages': none_like_pages,
    'empty_page_ratio': empty_page_ratio,
    'short_page_ratio': short_page_ratio,
    'deepseek_errors': deepseek_errors,
    'block_ocr_expected': block_ocr_expected,
    'block_ocr_outputs': block_ocr_outputs_count,
    'block_ocr_with_text': block_ocr_with_text,
    'block_ocr_text_coverage': block_ocr_text_coverage,
    'rag_chunks': rag_count,
    'spellcheck': spellcheck_meta if 'spellcheck_meta' in globals() else {'enabled': False},
}
quality_report['passed'] = not (
    broken_links
    or quality_report['pages_in_markdown'] != quality_report['pages_expected']
    or deepseek_errors
    or quality_report['duplicate_image_ids']
    or none_like_pages
    or (QUALITY_FAIL_ON_EMPTY_TEXT and empty_page_ratio > MAX_EMPTY_PAGE_RATIO)
    or short_page_ratio > MAX_SHORT_PAGE_RATIO
    or (RUN_BLOCK_OCR and block_ocr_expected > 0 and block_ocr_text_coverage < MIN_BLOCK_OCR_TEXT_COVERAGE)
)

(METADATA_DIR / 'quality_report.json').write_text(json.dumps(quality_report, ensure_ascii=False, indent=2), encoding='utf-8')
quality_lines = [
    '# Phase 1 Quality Report',
    '',
    f"- passed: `{quality_report['passed']}`",
    f"- pages: `{quality_report['pages_in_markdown']}/{quality_report['pages_expected']}`",
    f"- image records: `{quality_report['image_records']}`",
    f"- image links in markdown: `{quality_report['image_links_in_markdown']}`",
    f"- broken image links: `{len(broken_links)}`",
    f"- empty pages: `{len(empty_pages)}`",
    f"- short pages: `{len(short_pages)}`",
    f"- none-like pages: `{len(none_like_pages)}`",
    f"- empty page ratio: `{empty_page_ratio:.1%}`",
    f"- short page ratio: `{short_page_ratio:.1%}`",
    f"- block OCR text coverage: `{block_ocr_text_coverage:.1%}`",
    f"- block OCR: `{block_ocr_with_text}/{block_ocr_expected}` with text, outputs `{block_ocr_outputs_count}`",
    f"- DeepSeek errors: `{len(deepseek_errors)}`",
    f"- RAG chunks: `{rag_count}`",
]
(METADATA_DIR / 'quality_report.md').write_text('\n'.join(quality_lines) + '\n', encoding='utf-8')
summary_path = METADATA_DIR / 'book.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    summary['quality_report'] = quality_report
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(quality_report, ensure_ascii=False, indent=2))
if QUALITY_STRICT and not quality_report['passed']:
    raise RuntimeError('Quality audit failed. Xem metadata/quality_report.json.')


In [ ]:
from docx import Document
from docx.shared import Inches


def make_docx():
    doc = Document()
    doc.add_heading(INPUT_PDF.stem, level=1)
    for line in (OUTPUT_DIR / 'book.md').read_text(encoding='utf-8').splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith('> ') or stripped == '</break>':
            continue
        if stripped.startswith('# '):
            continue
        if stripped.startswith('## '):
            doc.add_heading(stripped[3:].strip(), level=2)
        elif stripped.startswith('!['):
            match = re.search(r'\]\((.*?)\)', stripped)
            if match:
                image_path = OUTPUT_DIR / match.group(1)
                if image_path.exists():
                    try:
                        doc.add_picture(str(image_path), width=Inches(5.6))
                    except Exception:
                        doc.add_paragraph(str(image_path))
        else:
            doc.add_paragraph(stripped)
    doc.save(str(OUTPUT_DIR / 'book.docx'))
    return True


def make_contact_sheet(records):
    records = [r for r in records if r.get('path')]
    if not records:
        return None
    thumb_w, label_h, pad, cols = 260, 42, 12, 3
    rows = math.ceil(len(records) / cols)
    sheet = Image.new('RGB', (cols * thumb_w + (cols + 1) * pad, rows * (thumb_w + label_h) + (rows + 1) * pad), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, rec in enumerate(records):
        path = OUTPUT_DIR / rec['path']
        if not path.exists():
            continue
        with Image.open(path).convert('RGB') as img:
            img.thumbnail((thumb_w, thumb_w - label_h), Image.Resampling.LANCZOS)
            col, row = idx % cols, idx // cols
            x = pad + col * (thumb_w + pad)
            y = pad + row * (thumb_w + label_h + pad)
            sheet.paste(img, (x + (thumb_w - img.width) // 2, y))
            draw.text((x + 4, y + thumb_w - 12), f'{rec.get("id")} p.{rec.get("page_number")}', fill=(20, 20, 20))
    out = PREVIEW_DIR / 'images_contact.jpg'
    sheet.save(out, quality=88)
    return out


print('docx:', make_docx())
contact = make_contact_sheet(image_records)
print('contact:', contact)
for path in [OUTPUT_DIR / 'book.md', OUTPUT_DIR / 'book.docx', OUTPUT_DIR / 'rag_chunks.jsonl', METADATA_DIR / 'book.json', METADATA_DIR / 'images.json', contact]:
    if path and Path(path).exists():
        print(path, Path(path).stat().st_size)


In [ ]:
# Tạo package gọn để tải về: không zip model, repo clone, page render và worker outputs nặng.
def should_include_in_package(path):
    rel = path.relative_to(OUTPUT_DIR)
    parts = rel.parts
    if not parts:
        return False
    top = parts[0]
    if top in {'_PDF-Extract-Kit', '_models_pdf_extract_kit', '_manifests'}:
        return False
    if top == 'pages' and not INCLUDE_RENDERED_PAGES_IN_ZIP:
        return False
    if top == 'text_crops' and not INCLUDE_RAW_WORKER_OUTPUTS_IN_ZIP:
        return False
    if top in {'_deepseek_pages', '_spellcheck_pages'} and not INCLUDE_RAW_WORKER_OUTPUTS_IN_ZIP:
        return False
    if top == '_deepseek_blocks' and not INCLUDE_RAW_WORKER_OUTPUTS_IN_ZIP:
        return False
    if len(parts) >= 2 and parts[0] == '_previews' and parts[1] == 'layout_boxes' and not INCLUDE_LAYOUT_BOXES_IN_ZIP:
        return False
    if path.suffix in {'.pt', '.pth', '.onnx', '.safetensors'}:
        return False
    if '__pycache__' in parts:
        return False
    return True


zip_path = OUTPUT_DIR.with_suffix('.zip')
if ZIP_OUTPUT:
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path.is_file() and should_include_in_package(path):
                archive.write(path, path.relative_to(OUTPUT_DIR.parent))
    print('lean zip:', zip_path, zip_path.stat().st_size)

try:
    from IPython.display import display, Image as IPImage
    contact = PREVIEW_DIR / 'images_contact.jpg'
    layout_files = sorted(LAYOUT_VIS_DIR.glob('*_layout.jpg'))
    if contact.exists():
        display(IPImage(filename=str(contact)))
    if layout_files:
        display(IPImage(filename=str(layout_files[0])))
except Exception as exc:
    print(exc)

if DELETE_INTERMEDIATE_DIRS_AFTER_ZIP:
    for rel in ['pages', 'text_crops', '_manifests', '_deepseek_pages', '_deepseek_blocks', '_spellcheck_pages']:
        target = OUTPUT_DIR / rel
        if target.exists():
            shutil.rmtree(target, ignore_errors=True)
            print('deleted intermediate:', target)
    layout_dir = PREVIEW_DIR / 'layout_boxes'
    if layout_dir.exists() and not INCLUDE_LAYOUT_BOXES_IN_ZIP:
        shutil.rmtree(layout_dir, ignore_errors=True)
        print('deleted intermediate:', layout_dir)

if DELETE_SHARED_CACHE_AFTER_RUN and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    print('deleted shared cache:', CACHE_DIR)

print('\nImportant files:')
for rel in [
    'book.md',
    'book_raw_ocr.md',
    'book_corrected.md',
    'book.docx',
    'rag_chunks.jsonl',
    'metadata/book.json',
    'metadata/images.json',
    'metadata/layout_detections.json',
    'metadata/quality_report.json',
    '_previews/images_contact.jpg',
]:
    path = OUTPUT_DIR / rel
    print(rel, 'OK' if path.exists() else 'MISSING', path.stat().st_size if path.exists() else '')

print('\nFinal output directory:')
print(OUTPUT_DIR)
print('Download zip:')
print(zip_path)
